# Feature research: when more predictors hurt temporal stability

**Question:** does a wider representation improve the original 700-feature
LightGBM control on later development periods? We tested 4,617 additional
hypotheses, retained 256 using earlier data, and completed four conditions
across all five expanding folds. The archived control contributes five reused
fits. Every condition covers the same 727,187 validation applications.

**Result:** the full engineered condition improves pooled AUC, average precision,
Brier and log loss, but reduces mean official stability from 0.585188 to 0.559904.
Doubling the original feature budget gives a small mean gain and a weaker worst
fold. These findings support retaining the frozen release; no result was promoted.

This is post-release exploratory development research. The original final holdout
was already opened and is not used here. The original tuning and blend decisions
also used these development periods; this is not a fresh independent evaluation.
Read [the frozen release](09_model_release.ipynb) for its separate final result.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from home_credit.modeling.feature_research_report import FAMILIES, LABELS, WIDTHS, load_evidence

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
evidence = load_evidence(root)
result, screen = evidence["result"], evidence["screen"]
print("Verified complete study:", result["study_key"])
print("20 new comparison fits + 5 reused controls; 2 separate early-screen model fits.")
print("This review performs zero fits and requires no private data or cloud credentials.")
display(
    pd.DataFrame(
        [
            {
                "Original candidates": result["original_candidates"],
                "Additional hypotheses": screen["generated"],
                "Additional retained": screen["retained"],
                "Additional rejected": screen["rejected"],
                "Cases per condition": 727187,
            }
        ]
    )
)
display(pd.Series(screen["rejection_counts"], name="Rejected candidates").to_frame())

## Formulas, selection and availability

The original 2,508 features summarize 17 relational groups. The extension adds
ratios, dispersion, recency shares, household comparisons, category interactions,
missingness, source-order differences and training-population peer statistics.
These total 7,125 representations, not independent signals. Every new candidate
has source columns, an operation, a rationale and a recorded rejection decision.

Structural screening uses only training rows. Target and drift screeners fit
weeks 0-24 (160,000 sampled cases) and evaluate weeks 25-32 (80,000 cases), before
model-validation weeks 33-72. After constants, duplicates and near-constants,
2,159 additions remain; missingness/cardinality filters leave 2,036 eligible.
A fixed ranking budget retains 256. Early-window drift AUC is 0.998473, making
subsequent temporal testing essential. Screen metrics are not final CV results.

Positive ratios return null for missing or nonpositive denominators; relative
standard deviation is `std / abs(mean)` when `abs(mean) > 1e-8`, otherwise null.
Category combinations receive
training-fold frequency encoding. Each model fits its own peer references:
44 selected empirical ranks, three conditional median ratios and three conditional
interquartile positions. Ranks mostly re-express existing information. Unsupported
peer groups (fewer than 50 training cases) and unseen groups return null.

Source group order is **not verified chronology**. Historical quantiles/skew were
not recomputed from raw histories; peer quantiles describe training populations.
All features rely on the competition's application-time snapshots, without a
production guarantee about event timestamps or label maturity. CatBoost's native
target statistics were tested in the original benchmark; online default-rate
histories are not introduced without outcome-availability timestamps.

In [ ]:
examples = [next(r for r in screen["selected"] if r["family"] == f) for f in FAMILIES]
with pd.option_context("display.max_colwidth", 110):
    display(pd.DataFrame(examples)[["family", "operation", "sources", "rationale"]])
display(
    pd.DataFrame(
        {"Candidates": result["candidate_families"], "Retained": result["retained_families"]}
    ).rename(index=FAMILIES)
)

## Controlled comparisons and model complexity

The four new conditions use the original LightGBM parameters, seeds, six-thread
configuration and five full folds. Only the representation changes. All 20 fits
retain native models, encoders, peer references, feature recipes and predictions.
The 700-feature control reuses its archived predictions. No learner was retuned
to rescue a weak feature condition.

Report mean and worst-fold stability alongside pooled ROC AUC, average precision,
raw Brier and log loss. Native-model size and selected boosting rounds expose
complexity; they are not inference-latency measurements. Different worker instance
families prevent treating job duration as a controlled speed comparison.

The official metric penalizes declining weekly Gini and residual variation.
Pooled discrimination can therefore improve while the primary temporal objective
worsens. Fold changes make that tradeoff visible. Permutation error bars show the
observed minimum/maximum across five folds and three repeats, **not confidence
intervals**. Permuting a whole added family within each week preserves its internal
relationships but breaks dependencies with unpermuted features. On 12,000 sampled
cases per fold, the slope penalty can produce wide ranges. Full-fold ablations
carry more weight than one noisy importance number.

In [ ]:
from home_credit.modeling.feature_research_report import display_charts

rows = pd.DataFrame(result["rows"])
rows.insert(1, "features", rows["experiment"].map(WIDTHS))
display(rows.round(6))
fits = pd.DataFrame(
    [
        {
            "condition": LABELS[r["experiment"]],
            "rounds": r["best_iteration"],
            "native_model_MB": r["artifacts"]["model"]["bytes"] / 1_000_000,
        }
        for r in result["fit_records"]
    ]
)
display(fits.groupby("condition").agg(["mean", "min", "max"]).round(3))
display_charts(evidence)

## Interpretation, redundancy and an audited correction

The first SHAP implementation selected a sorted sample prefix, which could
overrepresent earlier source-ordered cases. This publication uses the corrected
independent uniform sample of 512 cases from each complete validation fold.
All eight weeks are represented. A separate job replayed all 727,187 validation
predictions using saved native models, encoders and peer maps, with zero new fits.
It checked SHAP additivity and retained links to the original diagnostic hashes.

TreeSHAP values are raw log-odds contributions, averaged equally across the five
fold samples. The table shows variation and membership in each fold's top twenty.
Correlation pairs use up to 4,000 systematically spaced training cases, pairwise
complete values, and absolute correlation at least 0.98 among the 80 leading
gain-ranked encoded predictors. This is a targeted redundancy check, not a
complete dependence analysis.

Sex is the leading mean-absolute-SHAP predictor in this engineered condition;
birth-related features also rank highly. That is a disclosed modeling limitation,
not a fairness finding or justification for a credit decision. Interpretations
are predictive associations, not causal effects. The model card states that
this pipeline has not been validated for real lending decisions.

In [ ]:
from home_credit.modeling.feature_research_report import importance_summary

importance = importance_summary(evidence)
with pd.option_context("display.max_colwidth", 100):
    display(importance.head(20).round(6))
    added = importance[importance["name"].str.startswith("research__")].head(5)
    rationale = pd.DataFrame(screen["selected"])
    display(
        added.merge(rationale, on=["name", "family"])[
            ["name", "mean_abs_shap", "operation", "sources", "rationale"]
        ].round(6)
    )
    redundant = pd.DataFrame(evidence["redundancy"])
    redundant = redundant.sort_values(["training_correlation", "fold"], ascending=[False, True])
    display(redundant.head(12))
checks = [
    {
        k: d[k]
        for k in [
            "fold",
            "replayed_predictions",
            "prediction_maximum_absolute_error",
            "shap_rows",
            "maximum_additivity_absolute_error",
            "new_model_fits",
            "peer_references_refitted",
        ]
    }
    for d in evidence["diagnostics"]
]
display(pd.DataFrame(checks))
weeks = pd.DataFrame(
    [{"fold": d["fold"], **w} for d in evidence["diagnostics"] for w in d["shap_week_counts"]]
)
display(weeks.pivot(index="fold", columns="WEEK_NUM", values="len").fillna(0).astype(int))
print("Independent metric recomputation:", evidence["verification"]["metric_comparisons"], "checks")
print("Maximum metric absolute error:", evidence["verification"]["maximum_metric_absolute_error"])

## Decision and reproducible closure

- **All 256 additions:** mean stability falls by 0.025284 despite better pooled
  discrimination and probability scores. Feature influence alone does not justify
  inclusion in a model optimized for temporal stability.
- **Remove the 95 added ratios:** mean stability recovers by 0.024731 relative to
  the full extension, finishing 0.000553 below the control with a stronger worst
  fold. This is not a claim of statistical equivalence.
- **Remove 50 peer features:** mean stability falls another 0.004293 relative to
  the full extension, while its worst fold improves. Their utility is contextual.
- **Double the original budget:** mean rises by only 0.001290 and worst-fold
  stability falls by 0.031427, with twice the inputs. No significance claim follows.

**Decision: preserve the frozen tuned blend.** This extension has answered its
bounded research questions; it does not retroactively optimize the observed
holdout. Raw-history quantiles, neural challengers and fully nested promotion
studies remain possible future work, not unfinished requirements for this release.

Independent recomputation verified 21 prediction files, 3,635,935 predictions
across the five conditions and 235 metric identities. The original feature job
stopped at its time limit after 11 durable fits. Three workers supplied eight
disjoint late-fold fits; the collector resumed the one unfinished early-fold fit.
No completed model fit was repeated. The complete driver then restored all 20
checkpoints with **zero new fits**, and interpretation also demonstrated unchanged
zero-fit reuse. Recovery is at complete-fold boundaries, not mid-tree checkpoints.

This notebook validates committed aggregate evidence and can run without private
data. The catalog, comparison, ledgers, corrected diagnostics, verification script
and exact hashes are linked in [the research record](../reports/feature_research/README.md).
[Notebook 12](12_calibration.ipynb) separately tests temporal probability calibration.